# Домашнее задание: логистическая регрессия (health_data.xlsx)

**Таргет:** illness (болен / не болен). Обучить логистическую регрессию по остальным признакам.

1. Загрузить таблицу health_data  
2. Предобработка данных (пропуски, категориальные признаки)  
3. Разбить на фичи и таргет (illness), обучающую и валидационную выборки  
4. Стандартизация числовых полей  
5. Обучить модель логистической регрессии  
6. Метрики качества (accuracy, precision, recall, F1, ROC-AUC, матрица ошибок)  
7. Визуализация важности фич (коэффициенты)

In [ ]:
# Шаг 1: загрузка health_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel("health_data.xlsx", sheet_name="Sheet1")
print(df.shape)
df.head()

In [ ]:
# Шаг 2: предобработка — удалить строки с пропуском в таргете, заполнить пропуски в фичах
df = df.dropna(subset=["illness"])
df["illness"] = df["illness"].astype(int)
# Пропуски в числовых полях — медиана по столбцу
num_cols = ["age", "weight", "height", "chronic_diseases", "family_history"]
for c in num_cols:
    if c in df.columns and df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())
# Категориальные: smoking_status, exercise_frequency, diet — закодируем позже через OneHot
cat_cols = ["smoking_status", "exercise_frequency", "diet"]
df[cat_cols] = df[cat_cols].astype(str)
print("Пропуски после предобработки:", df.isna().sum().sum())
df.head()

In [ ]:
# Шаг 3: разбиение на фичи, таргет и обучающую/валидационную выборки
from sklearn.model_selection import train_test_split

target_col = "illness"
feature_cols_num = ["age", "weight", "height", "chronic_diseases", "family_history"]
feature_cols_cat = ["smoking_status", "exercise_frequency", "diet"]
X = df[feature_cols_num + feature_cols_cat]
y = df[target_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape[0], "Val:", X_val.shape[0])

In [ ]:
# Шаг 4: стандартизация числовых полей + кодирование категориальных (pipeline)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), feature_cols_num),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), feature_cols_cat)
])
X_train_tr = preprocessor.fit_transform(X_train)
X_val_tr = preprocessor.transform(X_val)
feature_names = preprocessor.get_feature_names_out()
print("Фичи после преобразования:", list(feature_names))

In [ ]:
# Шаг 5: обучение логистической регрессии
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tr, y_train)
print("Модель обучена.")

In [ ]:
# Шаг 6: метрики качества
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_val_tr)
y_proba = model.predict_proba(X_val_tr)[:, 1]

print("Accuracy:   ", round(accuracy_score(y_val, y_pred), 4))
print("Precision: ", round(precision_score(y_val, y_pred, zero_division=0), 4))
print("Recall:     ", round(recall_score(y_val, y_pred, zero_division=0), 4))
print("F1:         ", round(f1_score(y_val, y_pred, zero_division=0), 4))
print("ROC-AUC:   ", round(roc_auc_score(y_val, y_proba), 4))
print("\nМатрица ошибок:")
print(confusion_matrix(y_val, y_pred))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Шаг 7: визуализация важности фич (коэффициенты логистической регрессии)
coef = model.coef_[0]
importance = pd.Series(coef, index=feature_names).sort_values(key=abs, ascending=True)
importance.plot(kind="barh", figsize=(8, 6), edgecolor="black")
plt.title("Важность фич (коэффициенты логистической регрессии)")
plt.xlabel("Коэффициент")
plt.tight_layout()
plt.show()